<a href="https://colab.research.google.com/github/konerulalith-prog/llm-security-copilot/blob/main/LLM_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Downloading Dataset
!wget https://raw.githubusercontent.com/logpai/loghub/master/Linux/Linux_2k.log -O auth.log

--2026-03-15 22:31:40--  https://raw.githubusercontent.com/logpai/loghub/master/Linux/Linux_2k.log
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 216485 (211K) [text/plain]
Saving to: ‘auth.log’

auth.log            100%[===================>] 211.41K  --.-KB/s    in 0.06s   

2026-03-15 22:31:40 (3.52 MB/s) - ‘auth.log’ saved [216485/216485]



In [2]:
#Loading the logs
def load_logs(file_path):
    with open(file_path, "r") as file:
        logs = file.readlines()
    return logs

logs = load_logs("auth.log")
print("Total logs:", len(logs))

Total logs: 2000


In [3]:
#Filtering Suspicious Logs
def filter_logs(logs):
    keywords = [
        "Failed password",
        "invalid user",
        "Accepted password",
        "authentication failure",
        "sudo"
    ]
    return [log for log in logs if any(k in log for k in keywords)]

filtered_logs = filter_logs(logs)

print("Security related logs:", len(filtered_logs))

Security related logs: 490


In [4]:
#Showing suspicious entries
print("Sample suspicious entries:\n")

for log in filtered_logs[:5]:
    print(log.strip())

Sample suspicious entries:

Jun 14 15:16:01 combo sshd(pam_unix)[19939]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=218.188.2.4
Jun 14 15:16:02 combo sshd(pam_unix)[19937]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=218.188.2.4
Jun 15 02:04:59 combo sshd(pam_unix)[20882]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=220-135-151-1.hinet-ip.hinet.net  user=root
Jun 15 02:04:59 combo sshd(pam_unix)[20884]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=220-135-151-1.hinet-ip.hinet.net  user=root
Jun 15 02:04:59 combo sshd(pam_unix)[20883]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=220-135-151-1.hinet-ip.hinet.net  user=root


In [5]:
#Install Free LLM
!pip install --upgrade transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 21.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [6]:
from transformers import pipeline
import torch

generator = pipeline(
    "text-generation",
    model="tiiuae/falcon-rw-1b",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1
)

print("Instruction model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.62G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Instruction model loaded successfully.


In [7]:
def prepare_logs_for_llm(logs, limit=10):
    selected = logs[:limit]
    formatted = "\n".join([log.strip() for log in selected])
    return formatted

In [8]:
def analyze_logs(question, logs):

    failed_logs = [log for log in logs if "authentication failure" in log or "Failed password" in log]
    success_logs = [log for log in logs if "Accepted password" in log]

    failed_count = len(failed_logs)
    success_count = len(success_logs)

    question = question.lower()

    if "summarize" in question:
        return (
            f"The logs contain {failed_count} failed login attempts and "
            f"{success_count} successful logins. "
            "Repeated authentication failures suggest possible suspicious activity."
        )

    elif "brute" in question:
        if failed_count >= 5:
            return (
                f"There are {failed_count} failed login attempts. "
                "This volume of repeated failures may indicate a brute-force attack."
            )
        else:
            return (
                f"There are {failed_count} failed login attempts. "
                "This does not strongly indicate brute-force activity."
            )

    elif "repeated" in question:
        return (
            f"There are {failed_count} failed login attempts recorded in the logs, "
            "indicating repeated authentication failures."
        )

    else:
        return (
            f"There are {failed_count} failed login attempts and "
            f"{success_count} successful login events in the logs."
        )

In [9]:
def security_copilot_interface(logs):

    print("     LLM Security Copilot System   ")
    print("Ask security-related questions.")
    print("Type 'exit' to stop.\n")

    while True:
        user_query = input("Your Question: ")

        if user_query.lower() == "exit":
            print("\nExiting Security Copilot.")
            break

        response = analyze_logs(user_query, logs)

        print("\nSecurity Copilot Response:\n")
        print(response)


In [ ]:
security_copilot_interface(filtered_logs)

     LLM Security Copilot System   
Ask security-related questions.
Type 'exit' to stop.

Your Question: repeated login attempts

Security Copilot Response:

There are 490 failed login attempts recorded in the logs, indicating repeated authentication failures.
